# 농업 데이터 수집 자동화 Agent 예제

농촌진흥청·공공데이터포털 **파일데이터(CSV/ZIP)** 를 Agent Tool이 직접 내려받고
SQLite에 적재합니다. Open API 인증키(`serviceKey`)는 쓰지 않습니다.

```text
사용자: "포털에서 CSV를 내려받고 창고에 적재해줘"
        │
        ▼
   create_agent
    목록 / download_portal_data / 적재 / 조회 / 경보
        │
        ▼
   공공데이터포털 fileDownload.do
        │
        ▼
   data/*.csv → SQLite 창고
```

- OpenAI 키: `C:\env\.env` 의 `OPENAI_API_KEY`
- 포털 URL·임계값: `farm_collect_core.COLLECTION_CONFIG` (별도 JSON 파일 없음)
- `data/` 는 다운로드 캐시. 없어도 Tool이 다시 만든다.
- 기상: 2023년 전북 완주군 (전주 인근). 원본 약 85MB는 정리본이 있으면 생략
- 농가·생육: 2024년 노지 (고추, 밀, 배추, 양파, 콩 등)


## STEP 0. 환경 준비


In [11]:
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent

from farm_collect_core import (
    DATA_DIR,
    DB_PATH,
    SAMPLE_QUESTIONS,
    TOOL_LABELS,
    ask_agent,
    build_runtime,
    format_sources,
    load_config,
)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 60)

load_dotenv(r"C:\env\.env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(r"OPENAI_API_KEY 를 C:\env\.env 에서 찾을 수 없습니다.")
print("OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("데이터 폴더:", DATA_DIR)
print("창고 DB:", DB_PATH)
print("create_agent:", create_agent.__module__)


OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)
데이터 폴더: C:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\02_농업 데이터 수집 자동화 시스템 개발 실습\data
창고 DB: C:\env\farm_warehouse\farm_warehouse.db
create_agent: langchain.agents.factory


## STEP 1. 공공 CSV 수집 대상

포털 URL·제공기관·임계값은 `COLLECTION_CONFIG` 코드 상수입니다. `data/` JSON은 필요 없습니다.

Agent의 `download_portal_data` 가 아래 포털에서 파일을 받습니다. (로그인 없이 파일 다운로드)

- 기상 https://www.data.go.kr/data/15136768/fileData.do
- 노지 농가·생육 https://www.data.go.kr/data/15126334/fileData.do
- 예찰 지점 https://www.data.go.kr/data/15123424/fileData.do
- 병해충 목록 https://www.data.go.kr/data/15151253/fileData.do


In [12]:
cfg = load_config()
display(pd.DataFrame(cfg["sources"])[["source_id", "name", "file", "year", "provider", "portal"]])
print("초점 지역:", cfg["region_focus"])
print("경보 임계값:", cfg["alert_thresholds"])
print(format_sources())
print("\n[미리보기] CSV가 있으면 표시합니다. 없으면 시나리오 B에서 다운로드합니다.")
for name in ["rda_weather_wanju_2023.csv", "rda_farm_info_2024.csv", "rda_growth_2024.csv"]:
    path = DATA_DIR / name
    if not path.exists():
        print(f"\n{name}: 아직 없음 (download_portal_data 후 생성)")
        continue
    df = pd.read_csv(path, encoding="utf-8-sig")
    print(f"\n{name} {len(df)}행")
    display(df.head(3))


,source_id,name,file,year,provider,portal
0,weather,병해충발생예측 활용 농업기상,rda_weather_wanju_2023.csv,2023,농촌진흥청,https://www.data.go.kr/data/15136768/fileData.do
1,farm,노지 현장 농가정보,rda_farm_info_2024.csv,2024,농촌진흥청,https://www.data.go.kr/data/15126334/fileData.do
2,growth,노지 작물 생육조사,rda_growth_2024.csv,2024,농촌진흥청,https://www.data.go.kr/data/15126334/fileData.do
3,pest_sites,병해충 예찰조사 지점,rda_pest_sites.csv,2015-2024,농촌진흥청,https://www.data.go.kr/data/15123424/fileData.do
4,pest_info,병해충 목록,rda_pest_catalog.csv,2025,농림수산식품교육문화정보원,https://www.data.go.kr/data/15151253/fileData.do


초점 지역: {'sido': '전북', 'sigungu': '완주군', 'note': '전주 인근. 농업기상은 완주군 반교리·이서면 관측점(2023).'}
경보 임계값: {'humidity_high_pct': 85, 'rainfall_heavy_mm': 20, 'rainfall_sensor_outlier_mm': 400, 'tmax_hot_c': 33}
수집 초점: 전북 완주군 (전주 인근. 농업기상은 완주군 반교리·이서면 관측점(2023).)
제공: 농촌진흥청·공공데이터포털 파일데이터(CSV). Open API 인증키 없음.
있는 작목: 고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도, 논벼
없는 작목: 토마토, 딸기, 수박, 참외

[수집 대상 CSV]
- weather: 병해충발생예측 활용 농업기상 / rda_weather_wanju_2023.csv / 2023 / 원본 730건 / 국가농작물병해충관리 기상정보 중 완주군 일별 집계
  포털: https://www.data.go.kr/data/15136768/fileData.do
- farm: 노지 현장 농가정보 / rda_farm_info_2024.csv / 2024 / 원본 312건 / 노지 농가 작목·품종·정식·수확량
  포털: https://www.data.go.kr/data/15126334/fileData.do
- growth: 노지 작물 생육조사 / rda_growth_2024.csv / 2024 / 원본 10914건 / 고추(전국)와 전북 밀·배추·양파·콩 생육
  포털: https://www.data.go.kr/data/15126334/fileData.do
- pest_sites: 병해충 예찰조사 지점 / rda_pest_sites.csv / 2015-2024 / 원본 9804건 / 작목별 예찰포 위치·면적
  포털: https://www.data.go.kr/data/15123424/fileData.do
- pest_info: 병해충 목록 / rda_pest_catalog.csv

,sido,sigungu,station,obs_date,tmin_c,tmax_c,tavg_c,humidity_pct,rainfall_mm,wind_ms,leaf_wetness
0,전북,완주군,완주군 반교리,2023-01-01,-4.0,5.0,-0.25,72.50,0.0,2.25,NaN
1,전북,완주군,완주군 반교리,2023-01-02,-7.0,1.0,-3.54,65.21,0.0,2.23,NaN
2,전북,완주군,완주군 반교리,2023-01-03,-9.0,2.0,-3.83,74.79,0.0,1.86,NaN



rda_farm_info_2024.csv 312행


,year,sido,sigungu,farm_id,crop,variety,...,between_row_m,sow_date,plant_date,harvest_date,yield_total,note
0,2024,경기,안성,1,고추,뚝심칼탄,...,NaN,NaN,2024-04-29,2024-10-02,336.0,NaN
1,2024,경기,화성,2,고추,칼탄맥스,...,NaN,NaN,2024-05-01,2024-09-03,139.0,NaN
2,2024,경기,안성,3,고추,트리플X,...,NaN,NaN,2024-05-04,2024-10-13,303.0,NaN



rda_growth_2024.csv 10914행


,sido,sigungu,crop,farm_id,survey_date,plant_no,plant_height_cm,fruit_count,harvest_count,note
0,경기,안성,고추,1,2024-05-17,1,30.5,0.0,0.0,NaN
1,경기,안성,고추,1,2024-05-17,2,37.1,0.0,0.0,NaN
2,경기,안성,고추,1,2024-05-17,3,37.7,0.0,0.0,NaN


## STEP 2~5. create_agent (다운로드 Tool 포함)


In [13]:
runtime = build_runtime()
print("Agent 준비 / 초점 지역:", runtime["region"])
print("작목:", ", ".join(runtime["crops"]))
print("tools:", list(TOOL_LABELS))
print("다운로드는 시나리오 B에서 download_portal_data Tool이 수행합니다.")


Agent 준비 / 초점 지역: 완주군
작목: 고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도, 논벼
tools: ['list_data_sources', 'download_portal_data', 'ingest_farm_data', 'get_collection_status', 'query_collected_data', 'detect_farm_alerts']
다운로드는 시나리오 B에서 download_portal_data Tool이 수행합니다.


In [14]:
def show_run(item):
    print("=" * 72)
    print(item["label"])
    print("Q:", item["text"])
    out = ask_agent(runtime, item["text"])
    print("호출 Tool:", " → ".join(out["tools"]) if out["tools"] else "(없음)")
    for tr in out["trace"]:
        if tr["label"] == "결과":
            print(f"  [결과:{tr['name']}] {tr.get('preview', '')[:180]}")
        else:
            print(f"  → {tr['label']} {tr.get('args')}")
    print("\n[최종 답]\n")
    print(out["answer"])
    return out


## STEP 5. 실행 시나리오


In [15]:
out_a = show_run(SAMPLE_QUESTIONS[0])


A. 수집 대상
Q: 공공데이터포털에서 받아 자동 수집할 수 있는 농업 데이터 종류를 알려줘.
호출 Tool: list_data_sources
  → 수집대상 목록 {}
  [결과:list_data_sources] 수집 초점: 전북 완주군 (전주 인근. 농업기상은 완주군 반교리·이서면 관측점(2023).) 제공: 농촌진흥청·공공데이터포털 파일데이터(CSV). Open API 인증키 없음. 있는 작목: 고추, 마늘, 밀, 배추, 사과, 양파, 옥수수, 콩, 포도, 논벼 없는 작목: 토마토, 딸기, 수박, 참외  [수집 대상 CSV] 

[최종 답]

공공데이터포털에서 자동 수집할 수 있는 농업 데이터 종류는 다음과 같습니다:

1. **농업기상 (weather)**
   - 파일명: rda_weather_wanju_2023.csv
   - 연도: 2023
   - 내용: 병해충 발생 예측 활용 농업기상 데이터 (완주군 일별 집계)
   - [포털 링크](https://www.data.go.kr/data/15136768/fileData.do)

2. **노지 농가 정보 (farm)**
   - 파일명: rda_farm_info_2024.csv
   - 연도: 2024
   - 내용: 노지 농가의 작목, 품종, 정식, 수확량 정보
   - [포털 링크](https://www.data.go.kr/data/15126334/fileData.do)

3. **노지 작물 생육 조사 (growth)**
   - 파일명: rda_growth_2024.csv
   - 연도: 2024
   - 내용: 고추(전국) 및 전북의 밀, 배추, 양파, 콩 생육 데이터
   - [포털 링크](https://www.data.go.kr/data/15126334/fileData.do)

4. **병해충 예찰 조사 지점 (pest_sites)**
   - 파일명: rda_pest_sites.csv
   - 연도: 2015-2024
   - 내용: 작목별 예찰포 위치 및 면적 정보
   - [포털 링

In [16]:
out_b = show_run(SAMPLE_QUESTIONS[1])


B. 전체 수집
Q: 공공데이터포털에서 농업 CSV를 내려받고 창고에 적재한 뒤 수집 현황을 보고해줘.


c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\02_농업 데이터 수집 자동화 시스템 개발 실습\portal_csv.py:264: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  growth_df = pd.concat(frames, ignore_index=True)


호출 Tool: download_portal_data → ingest_farm_data → get_collection_status
  → 포털 CSV 다운로드 {'source': 'all'}
  [결과:download_portal_data] 공공데이터포털 파일 다운로드 (Open API 키 없음) - 농촌진흥청_병해충발생예측 활용 기상정보 https://www.data.go.kr/data/15136768/fileData.do   기상 원본 약 85MB 재다운로드 생략. 정리본 rda_weather_wanju_2023.csv 사용. 다시 받으려면 force=T
  → CSV 적재 {'source': 'all'}
  → 수집 현황 {}
  [결과:ingest_farm_data] weather 적재 완료: rda_weather_wanju_2023.csv → weather (730건) farm 적재 완료: rda_farm_info_2024.csv → farm (312건) growth 적재 완료: rda_growth_2024.csv → growth (10914건) pest_sites 적재 완료: rd
  [결과:get_collection_status] 창고: farm_warehouse.db weather 730건 기간 2023-01-01 ~ 2023-12-31 farm 312건 기간 2023-09-10 ~ 2024-10-17 growth 10914건 기간 2024-01-10 ~ 2024-11-08 pest_sites 9804건 기간 1981 ~ 2024 pest_inf

[최종 답]

공공데이터포털에서 농업 CSV 파일을 성공적으로 다운로드하고 창고에 적재하였습니다. 

### 적재 현황
- **기상 데이터 (weather)**: 730건 (기간: 2023-01-01 ~ 2023-12-31)
- **농가 데이터 (farm)**: 312건 (기간: 2023-09-10 ~ 2024-10-17)
- **생육 데이터 (growth)**: 10,914건 (기간: 2024-01-

In [17]:
out_c = show_run(SAMPLE_QUESTIONS[2])


C. 완주 기상과 고추 생육
Q: 2023년 8월 완주군 농업기상(강수·습도·기온)과 2024년 고추 생육(전국)을 짧게 정리해줘.
호출 Tool: query_collected_data → query_collected_data
  → 창고 조회 {'table': 'weather', 'sido': '전북', 'sigungu': '완주', 'start_date': '2023-08-01', 'end_date': '2023-08-31'}
  → 창고 조회 {'table': 'growth', 'crop': '고추', 'start_date': '2024-01-01', 'end_date': '2024-12-31'}
 전북,완주군,완주군 반교리,2023-_data] weather 62건 요약: 강수합(400mm미만) 1362.0mm, 최고기온 35.4℃, 평균습도 81.8% sido,sigungu,station,obs_date,tmin_c,tmax_c,tavg_c,humidity_pct,rainfall_mm,wind_ms,leaf_wetness
 경기,안성,고추,1,2024-05-17,1,30.5,0.0,0.0, 요약: 80건(최대 80건 표시) 초장 평균 30.9cm sido,sigungu,crop,farm_id,survey_date,plant_no,plant_height_cm,fruit_count,harvest_count,note

[최종 답]

2023년 8월 전북 완주군의 농업기상 데이터는 다음과 같습니다:
- **강수량**: 총 1362.0mm (400mm 미만)
- **최고 기온**: 35.4℃
- **평균 습도**: 81.8%

2024년 고추 생육 데이터는 다음과 같습니다:
- **초장 평균**: 30.9cm
- **조사 건수**: 80건

출처: 농촌진흥청/공공데이터포털.


In [18]:
out_d = show_run(SAMPLE_QUESTIONS[3])


D. 이상 점검
Q: 전북 완주 인근 기상과 노지 농가 자료를 기준으로 이상·병해 관련 사항을 점검해줘.


c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\02_농업 데이터 수집 자동화 시스템 개발 실습\portal_csv.py:264: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  growth_df = pd.concat(frames, ignore_index=True)


호출 Tool: download_portal_data → download_portal_data → ingest_farm_data → ingest_farm_data → detect_farm_alerts → query_collected_data
  → 포털 CSV 다운로드 {'source': 'weather'}
  → 포털 CSV 다운로드 {'source': 'farm'}
  [결과:download_portal_data] 공공데이터포털 파일 다운로드 (Open API 키 없음) - 농촌진흥청_병해충발생예측 활용 기상정보 https://www.data.go.kr/data/15136768/fileData.do   기상 원본 약 85MB 재다운로드 생략. 정리본 rda_weather_wanju_2023.csv 사용. 다시 받으려면 force=T
  [결과:download_portal_data] 공공데이터포털 파일 다운로드 (Open API 키 없음) - 농촌진흥청_노지 현장 농가 데이터 https://www.data.go.kr/data/15126334/fileData.do   포털 다운로드 raw_field_farm.zip (924173바이트) atchFileId=FILE_000000003623007   정리 
  → CSV 적재 {'source': 'weather'}
  → CSV 적재 {'source': 'farm'}
  [결과:ingest_farm_data] weather 적재 완료: rda_weather_wanju_2023.csv → weather (730건)
  [결과:ingest_farm_data] farm 적재 완료: rda_farm_info_2024.csv → farm (312건)
  → 이상 경보 {}
  → 창고 조회 {'table': 'pest_info'}
  [결과:detect_farm_alerts] 임계값: 습도>=85%, 강수>=20mm, 최고기온>=33℃ / 기상 연도 2023 완주, 농가 연도 2024 [고습] 임계 85% 이상 79일 (최

In [19]:
out_e = show_run(SAMPLE_QUESTIONS[4])


E. 토마토(자료 없음)
Q: 토마토 하우스 생육 데이터와 정식 후 물 관리 기록을 보여줘.
호출 Tool: (없음)

[최종 답]

토마토 하우스 생육 데이터는 현재 수집 가능한 CSV 파일에 포함되어 있지 않습니다. 다른 작물로 대체할 수 없으므로, 해당 데이터를 제공할 수 없습니다. 다른 작물에 대한 정보가 필요하시면 말씀해 주세요.


## STEP 6. 결과 정리


In [20]:
summary = pd.DataFrame([
    {"시나리오": "A", "기대한 Tool": "list_data_sources", "실제 호출 Tool": ", ".join(out_a["tools"]), "비고": "포털 CSV 목록"},
    {"시나리오": "B", "기대한 Tool": "download_portal_data, ingest_farm_data, get_collection_status", "실제 호출 Tool": ", ".join(out_b["tools"]), "비고": "포털 다운로드 후 적재"},
    {"시나리오": "C", "기대한 Tool": "query_collected_data", "실제 호출 Tool": ", ".join(out_c["tools"]), "비고": "2023 완주 기상 + 2024 고추 생육"},
    {"시나리오": "D", "기대한 Tool": "detect_farm_alerts", "실제 호출 Tool": ", ".join(out_d["tools"]), "비고": "임계값·농가 병해 비고"},
    {"시나리오": "E", "기대한 Tool": "자료 없음 명시", "실제 호출 Tool": ", ".join(out_e["tools"]), "비고": "토마토를 고추로 대체하지 않음"},
])
display(summary)


,시나리오,기대한 Tool,실제 호출 Tool,비고
0,A,list_data_sources,list_data_sources,포털 CSV 목록
1,B,"download_portal_data, ingest_farm_data, get_collection_s...","download_portal_data, ingest_farm_data, get_collection_s...",포털 다운로드 후 적재
2,C,query_collected_data,"query_collected_data, query_collected_data",2023 완주 기상 + 2024 고추 생육
3,D,detect_farm_alerts,"download_portal_data, download_portal_data, ingest_farm_...",임계값·농가 병해 비고
4,E,자료 없음 명시,,토마토를 고추로 대체하지 않음
